# 03 - Sources locales de contexte

Ce notebook exporte le registre des sources locales et télécharge les sources directes utilisables pour les labels faibles.


In [1]:
# Paramètres
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SOURCE_KEYS = ["geneve_sitg_amenagements_2roues"]
CONTEXT_OUTPUT_CRS = "EPSG:4326"

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Exécuter ce notebook depuis la racine du dépôt projet-slow-vaud.")


In [2]:
import sys

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from slowvaud.context import fetch_context_source, write_context_registry
from slowvaud.paths import ensure_data_tree, load_config


## Registre des sources


In [3]:
cfg = load_config()
paths = ensure_data_tree()
registry_path = write_context_registry()

registry_rows = []
for source_key, source in cfg["context_sources"].items():
    registry_rows.append({
        "source_key": source_key,
        "agglomeration": source["agglomeration"],
        "type": source["type"],
        "status": source["status"],
        "source_crs": source.get("source_crs", "[A COMPLETER]"),
        "output_crs": source.get("output_crs", "[A COMPLETER]"),
        "metadata_url": source.get("metadata_url"),
    })

print(f"Registre: {registry_path}")
pd.DataFrame(registry_rows)


Registre: /Users/marc-ed/BAS-local/repos/projet-slow-vaud/data/manifests/context_sources_registry.json


,source_key,agglomeration,type,status,source_crs,output_crs,metadata_url
0,geneve_sitg_amenagements_2roues,geneve,arcgis_feature_server,direct_download,EPSG:2056,EPSG:4326,https://sitg.ge.ch/donnees/otc-amenag-2roues
1,lausanne_viageo_amenagement_cyclable,lausanne,manual_download,manual_or_authenticated_download,EPSG:2056,[A COMPLETER],https://viageo.ch/md/eb867568-c511-4304-b902-1...
2,berne_local_cycle_infrastructure,berne,todo,a_rechercher,[A COMPLETER],[A COMPLETER],NaN
3,bale_local_cycle_infrastructure,bale,todo,a_rechercher,[A COMPLETER],[A COMPLETER],NaN
4,zurich_local_cycle_infrastructure,zurich,todo,a_rechercher,[A COMPLETER],[A COMPLETER],NaN


## Téléchargement des sources directes


In [4]:
context_rows = []
for source_key in SOURCE_KEYS:
    source = cfg["context_sources"][source_key]
    output = fetch_context_source(source_key, config=cfg)
    row = {
        "source_key": source_key,
        "agglomeration": source["agglomeration"],
        "output": str(output),
        "status": source["status"],
    }
    if output.suffix == ".geojson":
        gdf = gpd.read_file(output)
        crs = str(gdf.crs)
        expected_crs = source.get("output_crs", CONTEXT_OUTPUT_CRS)
        if crs != expected_crs:
            raise ValueError(f"CRS inattendu pour {output}: {crs}")
        row.update({"features": len(gdf), "crs": crs})
    context_rows.append(row)

pd.DataFrame(context_rows)


,source_key,agglomeration,output,status,features,crs
0,geneve_sitg_amenagements_2roues,geneve,/Users/marc-ed/BAS-local/repos/projet-slow-vau...,direct_download,6759,EPSG:4326
